# Paper 1 — Measurement Study: Multi-Tier LLM Inference Characterization

> **Note on scope.** This notebook analyzes the *operational characteristics*
> of a deployed multi-tier system from production event logs (`data/logs/*.jsonl`).
> The *policy comparison* (cost/quality Pareto across routing strategies) lives
> in `paper2_learned_routing.ipynb` and is driven by the canonical experiment
> artifact in `results/<timestamp>/`. The two notebooks are complementary:
> Paper 1 characterizes what the live system does; Paper 2 evaluates which
> policy would be best.

**Target venues:** IEEE Access, ICSE-SEIP, ASE


## 1. Load Inference Events

Loads JSONL event logs from `data/logs/` (one file per day, produced by
`monitor/event_logger.py`). If no logs are present, falls back to a synthetic
event generator solely so this notebook is browseable on a fresh checkout —
**figures from synthetic data are clearly marked and must not appear in the
paper.**


In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

LOG_DIR = Path('../data/logs')
SYNTHETIC = False

events = []
for f in sorted(LOG_DIR.glob('inference_events_*.jsonl')):
    with open(f) as fp:
        for line in fp:
            events.append(json.loads(line))

if not events:
    SYNTHETIC = True
    print('No production logs found — generating SYNTHETIC events for notebook preview.')
    print('Any figures below are NOT paper-eligible until real logs are present.')
    rng = np.random.default_rng(42)
    n = 2000
    final_tier = rng.choice([1, 2, 3], size=n, p=[0.55, 0.30, 0.15])
    events = []
    for i in range(n):
        tier = int(final_tier[i])
        cost = {1: 0.0001, 2: 0.001, 3: 0.01}[tier] * rng.uniform(0.5, 1.5)
        latency = {1: 50, 2: 300, 3: 800}[tier] * rng.uniform(0.7, 1.5)
        events.append({
            'event_id': f'syn-{i:04d}',
            'final_tier': tier,
            'total_cost_usd': cost,
            'total_latency_ms': latency,
            'success': bool(rng.random() > 0.05),
            'confidence': float(rng.uniform(0.5, 0.99)),
            'failure_mode': 'none' if rng.random() > 0.08 else rng.choice(
                ['timeout', 'rate_limit', 'parse_error', 'semantic_failure']),
            'has_disagreement': bool(rng.random() < 0.18),
        })

df = pd.DataFrame(events)
print(f'Loaded {len(df)} events. SYNTHETIC = {SYNTHETIC}.')
df.head()


## 2. Tier Distribution

**Research question:** What fraction of requests are absorbed by lower tiers?


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['final_tier'].value_counts().sort_index()
ax.bar(counts.index.astype(str), counts.values,
       color=['#9bd99b', '#f9c069', '#e88989'])
ax.set_xlabel('Final tier')
ax.set_ylabel('Number of requests')
ax.set_title(f"Tier distribution (n={len(df)}) {'[SYNTHETIC]' if SYNTHETIC else ''}")
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v}\n({v/len(df):.1%})', ha='center', va='bottom')
plt.tight_layout()
plt.show()


## 3. Failure Mode Analysis

**Research question:** What are the dominant failure modes per tier?


In [ ]:
fail = df[df['failure_mode'] != 'none']
if len(fail) == 0:
    print('No failures recorded in this dataset.')
else:
    pivot = fail.groupby(['final_tier', 'failure_mode']).size().unstack(fill_value=0)
    if pivot.empty or pivot.values.sum() == 0:
        print(f'Pivot is empty ({len(fail)} failure rows; no plottable data). '
              f'Sample failure rows:')
        print(fail[['final_tier', 'failure_mode']].head())
    else:
        pivot.plot(kind='bar', stacked=True, figsize=(8, 4), colormap='tab10')
        plt.title(f"Failure modes by tier {'[SYNTHETIC]' if SYNTHETIC else ''}")
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()
        try:
            display(pivot)
        except NameError:
            print(pivot)


## 4. Cost / Quality Pareto Frontier (from canonical experiment)

This section **does not** re-run experiments. It loads the latest canonical
result emitted by `python -m python_core.scripts.run_experiment` and reproduces
the Pareto plot from that artifact. If you want a different artifact, set
`RESULTS_DIR` explicitly below.

For the original log-only Pareto analysis (each request's cost vs confidence,
not router-vs-router), see the supplementary cell at the end of this section.


In [ ]:
RESULTS_ROOT = Path('../results')
runs = sorted(RESULTS_ROOT.glob('*/')) if RESULTS_ROOT.exists() else []
if not runs:
    print('No experiment artifacts found under ../results/.')
    print('Generate one with: python -m python_core.scripts.run_experiment')
else:
    RESULTS_DIR = runs[-1]
    pareto = pd.read_csv(RESULTS_DIR / 'pareto.csv')
    manifest = json.loads((RESULTS_DIR / 'manifest.json').read_text())

    fig, ax = plt.subplots(figsize=(8, 6))
    frontier = pareto[pareto['on_frontier'] == 1].sort_values('cost_mean')
    if len(frontier) >= 2:
        ax.plot(frontier['cost_mean'], frontier['quality_mean'],
                '--', color='tab:green', alpha=0.6, label='Pareto frontier')
    for _, r in pareto.iterrows():
        on_f = bool(r['on_frontier'])
        ax.errorbar(r['cost_mean'], r['quality_mean'],
                    xerr=r['cost_ci'], yerr=r['quality_ci'],
                    fmt='o' if on_f else 'x',
                    color='tab:green' if on_f else 'tab:gray',
                    markersize=8, capsize=3)
        ax.annotate(r['router'], (r['cost_mean'], r['quality_mean']),
                    textcoords='offset points', xytext=(6, 6), fontsize=9)
    ax.set_xlabel('Cost per request (USD)')
    ax.set_ylabel('Quality (reward-model score)')
    ax.set_title(f"Pareto frontier — {manifest['experiment']['n_seeds']} seeds, 95% CI")
    ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Source: {RESULTS_DIR}/pareto.csv (dataset SHA-256 {manifest['dataset']['sha256'][:16]}...)")


### 4a. Supplementary: per-request cost vs confidence (log-only)

This is the original log-based analysis. It shows the distribution of cost and
self-reported confidence across **individual requests**, not router strategies.
The router-vs-router comparison above is the paper-eligible result.


In [ ]:
ok = df[df['success'] == True]
fig, ax = plt.subplots(figsize=(7, 5))
for tier in sorted(ok['final_tier'].unique()):
    sub = ok[ok['final_tier'] == tier]
    ax.scatter(sub['total_cost_usd'], sub['confidence'], alpha=0.4,
               label=f'Tier {tier} (n={len(sub)})', s=15)
ax.set_xlabel('Cost per request (USD)')
ax.set_ylabel('Self-reported confidence')
ax.set_xscale('log')
ax.set_title(f"Cost vs confidence by tier {'[SYNTHETIC]' if SYNTHETIC else ''}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Summary Table (paper Section IV)


In [ ]:
summary = df.groupby('final_tier').agg(
    n_requests=('event_id', 'count'),
    avg_cost=('total_cost_usd', 'mean'),
    avg_latency=('total_latency_ms', 'mean'),
    success_rate=('success', 'mean'),
).round({'avg_cost': 5, 'avg_latency': 0, 'success_rate': 3})
summary['share'] = (summary['n_requests'] / summary['n_requests'].sum()).round(3)
summary


## 6. Cost Savings

Headline claim: **X% of requests can be handled at Tier 1, saving Y% of costs**
versus an always-premium baseline. The cost-savings number here is computed
from production logs; the policy comparison that produces this distribution
lives in `paper2_learned_routing.ipynb`.


In [ ]:
actual = df['total_cost_usd'].sum()
premium_only_per_call = 0.01
always_premium = premium_only_per_call * len(df)
print(f"Actual system spend:     ${actual:.4f}")
print(f"Always-premium baseline: ${always_premium:.4f}")
print(f"Savings vs premium:      {(1 - actual/always_premium):.1%}")
tier1_share = (df['final_tier'] == 1).mean()
print(f"Share routed at Tier 1:  {tier1_share:.1%}")
